In [1]:
"""
4가지 데이터셋 생성 스크립트
- EDA_1st_result.csv : 원본 1st data (레벨값)
- Derived_Variable.csv : 1st data가 1차 차분된 값 + 파생변수들 (oil_diff_target 포함)

생성되는 데이터셋 (모두 타겟 변수는 oil_diff_target = OilPrice.diff().shift(-1)):
1) dataset1_raw_only        : 1st data 레벨값 + oil_diff_target (OilPrice 제외)
2) dataset2_raw_plus_derived: 1st data 레벨값 + 파생변수 + oil_diff_target (OilPrice 제외)
3) dataset3_diff_only       : 1st data 1차 차분 + oil_diff_target (차분된 OilPrice 제외)
4) dataset4_derived_full    : Derived_Variable 그대로 (차분된 1st + 파생변수 + target)
"""

import pandas as pd
from pathlib import Path

# ===== 경로 설정 =====
INPUT_DIR = Path("../data/Finance_Final")
OUTPUT_DIR = Path("../data/Finance_Final")  # 사용자 지정 상대경로
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDA_PATH = INPUT_DIR / "EDA_1st_result.csv"
DERIVED_PATH = INPUT_DIR / "Derived_Variable.csv"

TARGET_COL = "oil_diff_target"  # OilPrice 대신 사용할 타겟 변수


# ===== 데이터 로드 =====
df_raw = pd.read_csv(EDA_PATH, parse_dates=["date"])
df_derived = pd.read_csv(DERIVED_PATH, parse_dates=["date"])

print(f"[원본 1st data]      shape={df_raw.shape}, "
      f"기간={df_raw['date'].min().date()} ~ {df_raw['date'].max().date()}")
print(f"[Derived_Variable]   shape={df_derived.shape}, "
      f"기간={df_derived['date'].min().date()} ~ {df_derived['date'].max().date()}")


# ===== 컬럼 분류 =====
raw_cols = [c for c in df_raw.columns if c != "date"]

# 파생변수 컬럼 (Derived에만 있는 것)
derived_only_cols = [c for c in df_derived.columns if c not in df_raw.columns]

# Derived 안에서 1st data와 동일한 이름의 컬럼 (이미 1차 차분된 값)
diff_cols_in_derived = [c for c in df_raw.columns
                        if c in df_derived.columns and c != "date"]


def reorder_target_first(df: pd.DataFrame) -> pd.DataFrame:
    """date → oil_diff_target → 나머지 순으로 컬럼 정렬"""
    others = [c for c in df.columns if c not in ("date", TARGET_COL)]
    return df[["date", TARGET_COL] + others]


# ===== 1) 1st data만 (OilPrice → oil_diff_target 으로 교체) =====
dataset1 = df_raw.merge(
    df_derived[["date", TARGET_COL]],
    on="date",
    how="inner",
)
dataset1 = dataset1.drop(columns=["OilPrice"])
dataset1 = reorder_target_first(dataset1)


# ===== 2) 1st data + 파생변수 (OilPrice → oil_diff_target 으로 교체) =====
# 파생변수에 이미 oil_diff_target 포함되어 있음
dataset2 = df_raw.merge(
    df_derived[["date"] + derived_only_cols],
    on="date",
    how="inner",
)
dataset2 = dataset2.drop(columns=["OilPrice"])
dataset2 = reorder_target_first(dataset2)


# ===== 3) 1st data의 1차 차분만 (차분된 OilPrice → oil_diff_target 으로 교체) =====
# Derived에서 1st data와 동명 컬럼들(차분값) + oil_diff_target 만 사용
dataset3 = df_derived[["date"] + diff_cols_in_derived + [TARGET_COL]].copy()
dataset3 = dataset3.drop(columns=["OilPrice"])  # 차분된 OilPrice도 제거
dataset3 = reorder_target_first(dataset3)


# ===== 4) Derived_Variable 그대로 =====
dataset4 = df_derived.copy()
dataset4 = reorder_target_first(dataset4)


# ===== 저장 =====
out_files = {
    "dataset1_raw_only.csv":         dataset1,
    "dataset2_raw_plus_derived.csv": dataset2,
    "dataset3_diff_only.csv":        dataset3,
    "dataset4_derived_full.csv":     dataset4,
}

print(f"\n===== 저장 경로: {OUTPUT_DIR.resolve()} =====")
for fname, df in out_files.items():
    fpath = OUTPUT_DIR / fname
    df.to_csv(fpath, index=False)
    print(f"{fname:35s}  shape={df.shape}")


# ===== 요약 =====
print("\n===== 데이터셋 설명 (모두 타겟: oil_diff_target) =====")
print("1) dataset1_raw_only         : 1st data 레벨값 (OilPrice 제외) + oil_diff_target")
print("2) dataset2_raw_plus_derived : 1st data 레벨값 (OilPrice 제외) + 파생변수 (target 포함)")
print("3) dataset3_diff_only        : 1st data 1차 차분 (차분된 OilPrice 제외) + oil_diff_target")
print("4) dataset4_derived_full     : 1st data 1차 차분 + 파생변수 (target 포함)")

# 컬럼 확인용 출력
print(f"\n[dataset1 컬럼] {list(dataset1.columns)}")
print(f"[dataset2 컬럼] {list(dataset2.columns)}")
print(f"[dataset3 컬럼] {list(dataset3.columns)}")
print(f"[dataset4 컬럼] {list(dataset4.columns)}")

[원본 1st data]      shape=(4820, 14), 기간=2007-01-02 ~ 2026-03-16
[Derived_Variable]   shape=(4799, 28), 기간=2007-02-01 ~ 2026-03-16

===== 저장 경로: C:\Users\ekf44\Desktop\baf\BAF-26-1-finance_2\data\Finance_Final =====
dataset1_raw_only.csv                shape=(4799, 14)
dataset2_raw_plus_derived.csv        shape=(4799, 28)
dataset3_diff_only.csv               shape=(4799, 13)
dataset4_derived_full.csv            shape=(4799, 28)

===== 데이터셋 설명 (모두 타겟: oil_diff_target) =====
1) dataset1_raw_only         : 1st data 레벨값 (OilPrice 제외) + oil_diff_target
2) dataset2_raw_plus_derived : 1st data 레벨값 (OilPrice 제외) + 파생변수 (target 포함)
3) dataset3_diff_only        : 1st data 1차 차분 (차분된 OilPrice 제외) + oil_diff_target
4) dataset4_derived_full     : 1st data 1차 차분 + 파생변수 (target 포함)

[dataset1 컬럼] ['date', 'oil_diff_target', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TermSpread', 'TreasuryYield', 'FedFundsRate']
[d